This notebook pulls raw data from Snowflake and store in s3, similar to the procedures in `load_data.py`

In [1]:
import json
import os
import typing as t
from datetime import datetime

import numpy as np
import pandas as pd
from s3fs import S3FileSystem

# from pre_90_pd.utilities import config
from pre_90_pd.utilities.dao import DataWarehouseConnection
from pre_90_pd.utilities.functions import split_ids_by_range, timestamp_str

from toast_cap import LOGGER
from toast_cap.utilities import config
from toast_cap.utilities.dao import DataWarehouseConnection
from toast_cap.utilities.functions import split_ids_by_range, timestamp_str

ModuleNotFoundError: No module named 'pre_90_pd'

In [ ]:
## change values in this cell depending on your requirements
run_date_str = '20231010'  # queries will pull data up to 1 day prior to run_date_str
volume_start_dt = '20170101' # pull daily revenue data starting from this date
output_dir = f's3://toast-datascience-sandbox/PradeepA/pre90_pd_model/{run_date_str}' # directory to store the raw data
transaction_output_dir = os.path.join(output_dir, 'daily_data')  # directory to store the daily revenue data parquet files
os.environ["SNOWFLAKE_PRIVATE_KEY_PASSWORD"] = ""

In [2]:
partial_days = 1
num_days_volume_data = (pd.to_datetime(run_date_str) - pd.to_datetime(volume_start_dt)).days + 1 - partial_days

datetime_now = datetime.strptime(run_date_str, "%Y%m%d")
end_date = timestamp_str("%Y%m%d", days=partial_days, time_now=datetime_now)

ts_start_date = timestamp_str(
    "%Y%m%d",
    days=partial_days + num_days_volume_data - 1,
    time_now=datetime_now
)

NameError: name 'run_date_str' is not defined

In [4]:
## acct data
sql_acct = """
        WITH first_loan as (
        SELECT TOASTORDERS_RESTAURANT_ID, 
        MIN(created_date) AS first_loan_date,
        MIN(case when term_in_days=90 then created_date else NULL end) AS first_loan_date_90d,
        MIN(case when term_in_days=270 then created_date else NULL end) AS first_loan_date_270d,
        MIN(case when term_in_days=360 then created_date else NULL end) AS first_loan_date_360d
        FROM toast_capital.product_record
        WHERE product_type in ('FIXED_FEE_LOAN', 'FIXED_FEE_WINTER_LOAN', 'MCA')
        AND funding_partner IN ('Toast', 'WebBank')
        AND created_date <= TO_DATE(TO_VARCHAR({}), 'YYYYMMDD')
        GROUP BY 1
        )
        
        SELECT CAST(cust.ACCOUNT_TOAST_ID AS int) AS "rid"
            ,cust.TOASTORDERS_STATE AS "state"
            ,cust.CUSTOMER_STATUS AS "customer_status"
            ,cust.FIRST_ORDER_DATE AS "first_order_date"
            ,cust.CHURN_DATE AS "churn_date"
            ,cust.CHURN_REASON AS "Churn Reason"
            ,cust.account_restaurant_category as "account_restaurant_category"
            ,cust.restaurant_type as "restaurant_type"
            ,cust.parent_market_segment as "parent_market_segment"
            ,tx.MIN_DATE as "tx_date_min"
            ,tx.MAX_DATE as "tx_date_max"
            ,tx.GPV_MIN_DATE as "gpv_date_min"
            ,tx.GPV_MAX_DATE as "gpv_date_max"
            ,l.first_loan_date as "first_loan_date"
            ,l.first_loan_date_90d as "first_loan_date_90d"
            ,l.first_loan_date_270d as "first_loan_date_270d"
            ,l.first_loan_date_360d as "first_loan_date_360d"
        FROM ANALYTICS_CORE.CUSTOMER cust
        LEFT JOIN (
            SELECT op.TOASTORDERS_RESTAURANT_ID
                ,to_date(to_char(min(op.date_id)), 'YYYYMMDD') AS MIN_DATE
                ,to_date(to_char(max(op.date_id)), 'YYYYMMDD') AS MAX_DATE
                ,to_date(to_char(min(CASE WHEN op.PAYMENT_TYPE = 'CREDIT'
                                     THEN op.date_id ELSE NULL END)), 'YYYYMMDD') AS GPV_MIN_DATE
                ,to_date(to_char(max(CASE WHEN op.PAYMENT_TYPE = 'CREDIT'
                                     THEN op.date_id ELSE NULL END)), 'YYYYMMDD') AS GPV_MAX_DATE
            FROM PRODUCT_TRANSACTIONS.TOAST_TRANSACTION_VOLUME_HOURLY op
            WHERE op.TOASTORDERS_RESTAURANT_ID IS NOT NULL
            AND op.date_id between {} and {}
            GROUP BY op.TOASTORDERS_RESTAURANT_ID
            ) AS tx ON cust.ACCOUNT_TOAST_ID = tx.TOASTORDERS_RESTAURANT_ID
        LEFT JOIN first_loan l on cust.ACCOUNT_TOAST_ID = l.TOASTORDERS_RESTAURANT_ID 
        WHERE cust.FIRST_ORDER_DATE IS NOT NULL
            AND cust.ACCOUNT_TOAST_ID IS NOT NULL
"""

In [5]:
# accts = DataWarehouseConnection().query(sql_acct.format(end_date, ts_start_date, end_date))

accts = DataWarehouseConnection().query(sql_acct.format(end_date, ts_start_date, end_date))
for x in ['first_loan_date', 'first_loan_date_90d', 'first_loan_date_270d', 'first_loan_date_360d',
         'first_order_date', 'churn_date', 'tx_date_min', 'tx_date_max', 'gpv_date_min', 'gpv_date_max']:
    accts[x] = pd.to_datetime(accts[x])
accts.to_parquet(os.path.join(output_dir, 'accts.parquet'))

In [6]:
## module data
sql_module = '''
            SELECT CAST(r.ACCOUNT_TOAST_ID AS int) AS "restaurant_id",
                CAST(TO_VARCHAR(d.FIRSTDAYOFMONTH, 'YYYYMM') AS int) AS "yearmon",
                m.LIVE_ONLINE_ORDERING_MODULE_COUNT AS "live_online_ordering_module_count",
                m.live_saas_mrr AS "live_saas_mrr",
                m.live_saas_module_count AS "live_saas_module_count",
                m.live_gift_card_module_count AS "live_gift_card_module_count"
            FROM ANALYTICS_CORE_ARR.MONTHLY_CUSTOMER_LIVE_MODULES m
            INNER JOIN ANALYTICS_CORE.CUSTOMER r ON m.customer_id = r.customer_id
            INNER JOIN (SELECT DISTINCT YR_MO, FIRSTDAYOFMONTH
                        FROM ANALYTICS_CORE.DATE_DIM
                        WHERE DATE_ID between {} and {}) as d
            ON d.YR_MO = m.YR_MO
            WHERE r.FIRST_ORDER_DATE IS NOT NULL
            AND r.ACCOUNT_TOAST_ID IS NOT NULL;
'''

In [7]:
monthly_modules = DataWarehouseConnection().query(sql_module.format(ts_start_date, end_date))
monthly_modules['live_saas_mrr'] = monthly_modules['live_saas_mrr'].astype(float)
monthly_modules.rename(columns={'restaurant_id': config.GROUP}, inplace=True)
monthly_modules.to_parquet(os.path.join(output_dir, 'monthly_modules.parquet'))

In [8]:
## daily revenue data
rid_ls = accts[config.GROUP].tolist()
rid_group_dict = split_ids_by_range(rid_ls)
for k, v in rid_group_dict.items():
    LOGGER.info('...loading rid chunk %s', k)
    daily_ = DataWarehouseConnection().get_daily_rev(
        start_yyyymmdd=ts_start_date,
        end_yyyymmdd=end_date,
        rest_ids=list(map(str, v)),
    )
    daily_.rename(columns={'restaurant_id': config.GROUP}, inplace=True)
    if len(daily_) > 0:
        LOGGER.info('...saving rid chunk %s', k)
        daily_.to_parquet(os.path.join(transaction_output_dir, f'rid-{k}.parquet'))

...loading rid chunk 170000000000000000
...saving rid chunk 170000000000000000
...loading rid chunk 150000000000000000
...saving rid chunk 150000000000000000
...loading rid chunk 90000000000000000
...saving rid chunk 90000000000000000
...loading rid chunk 40000000000000000
...saving rid chunk 40000000000000000
...loading rid chunk 70000000000000000
...saving rid chunk 70000000000000000
...loading rid chunk 110000000000000000
...saving rid chunk 110000000000000000
...loading rid chunk 50000000000000000
...saving rid chunk 50000000000000000
...loading rid chunk 10000000000000000
...saving rid chunk 10000000000000000
...loading rid chunk 130000000000000000
...saving rid chunk 130000000000000000
...loading rid chunk 100000000000000000
...saving rid chunk 100000000000000000
...loading rid chunk 30000000000000000
...saving rid chunk 30000000000000000
...loading rid chunk 120000000000000000
...saving rid chunk 120000000000000000
...loading rid chunk 80000000000000000
...saving rid chunk 80000

In [9]:
## The following two queries may not be relevant for your project

## get Capital application dates for model validation
sql_capital_validation = """
SELECT entity_identifier as "entity_identifier", 
    cast(toastorders_restaurant_id as int) as "rid",
    TO_DATE(funding_requested_date_time) as "funding_requested_date",
    term as "term",
    is_pre_90
FROM toast_capital.loan_application_record
WHERE funding_requested_date_time is not null 
    AND funding_requested_date_time>='2022-01-01'
    AND toastorders_restaurant_id is not null
    --AND is_pre_90 is True
    ;
"""

In [10]:
df_capital_validation = DataWarehouseConnection().query(sql_capital_validation)
df_capital_validation['funding_requested_date'] = pd.to_datetime(df_capital_validation['funding_requested_date'])
df_capital_validation.to_parquet(os.path.join(output_dir, 'capital_validation_population.parquet'))

In [11]:
## Capital loan default records
sql_tc_default=  """
SELECT cast(pr.toastorders_restaurant_id as int) AS "rid",
       df.date_of_default as "tc_default_date", 
       df.default_reason as "tc_default_reason"
FROM toast_capital.defaulted_accounts df
JOIN toast.toast_capital.product_record pr
ON df.salesforce_opportunityid=pr.external_id
JOIN ANALYTICS_CORE.DATE_DIM as d
ON df.date_of_default = d.dt
WHERE
    d.date_id <= {}
;
"""

In [12]:
tc_default = DataWarehouseConnection().query(sql_tc_default.format(end_date))

tc_default['tc_default_date'] = pd.to_datetime(tc_default['tc_default_date'])
tc_default['tc_default_reason'] = tc_default['tc_default_reason'].str.lower().str.strip()
tc_default = tc_default.drop_duplicates()

tc_default.to_parquet(os.path.join(output_dir, 'tc_default.parquet'))

In [13]:
tc_default['tc_default_reason'].value_counts()

30 days no processing                 1092
close of business                      398
debt stacking                          191
change of location                      66
change of control                       57
bankruptcy                              52
change of legal entity                  48
change of processor                     23
breach of toast merchant agreement       7
insolvent                                5
competitive churn                        3
other                                    2
change of 25% owner                      2
debt burden                              2
diversion of collection                  1
not in good standing                     1
fraud                                    1
Name: tc_default_reason, dtype: int64